# Modeling 5: 1,000 California red/white training reviews

This stage expands the population from **California Cabernet Sauvignon versus Chardonnay** (Model 4) to **all confidently labeled California red and white wines**. The four classifiers remain Naive Bayes, logistic regression, linear SVM, and fine-tuned DistilBERT. They use only `description`; wine metadata selects the population and supplies labels, never model features.

All models train on the **same balanced 1,000 reviews** (500 red, 500 white) sampled from saved `train.parquet`. They are evaluated on **every eligible California red/white review** in saved `validation.parquet`, with its natural class balance. The saved `test.parquet` is never loaded. This new validation population is broader than Model 4's, so scores are not a pure training-size comparison.

Run the TF-IDF models locally or in Colab. DistilBERT needs PyTorch, Transformers, and Accelerate; a Colab GPU is recommended. If the package-check cell shows missing packages, run its commented install line once, **restart the kernel**, then rerun from the top. DistilBERT saves temporary checkpoints in `modeling_5_bert_checkpoints/`.

## 1. Load the saved partitions

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
LABELS = ['red', 'white']
if not all(Path(name).exists() for name in ['train.parquet', 'validation.parquet']):
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Place train.parquet and validation.parquet beside this notebook.') from exc
    print('Upload train.parquet and validation.parquet together.')
    files.upload()
assert all(Path(name).exists() for name in ['train.parquet', 'validation.parquet'])
saved_train = pd.read_parquet('train.parquet')
saved_validation = pd.read_parquet('validation.parquet')
assert saved_train.index.intersection(saved_validation.index).empty
print(f'Saved partitions: {len(saved_train):,} train; {len(saved_validation):,} validation')

Saved partitions: 101,506 train; 33,835 validation


## 2. Select the California population, the balanced training sample, and all validation reviews

Labels come from the existing `color` column. Remove empty reviews, label conflicts, and duplicate text after whitespace/case normalization. If a review appears in both saved partitions after this normalization, remove its **training** copy and retain the entire validation set. The 1,000-review sample is newly drawn from the broader California population; Model 4's narrower sample is not reused.

In [2]:
N_TRAIN = 1000
MODEL_NUMBER = 5

def california_pool(frame):
    pool = frame.loc[frame['province'].eq('California') & frame['color'].isin(LABELS)].dropna(subset=['description', 'color']).copy()
    pool['description_key'] = pool['description'].str.replace(r'\s+', ' ', regex=True).str.strip().str.casefold()
    pool = pool.loc[pool['description_key'].ne('')]
    conflicts = pool.groupby('description_key')['color'].nunique()
    pool = pool.loc[~pool['description_key'].isin(conflicts[conflicts > 1].index)]
    return pool.drop_duplicates('description_key')

train_pool = california_pool(saved_train)
validation_all = california_pool(saved_validation)
validation_keys = set(validation_all['description_key'])
cross_split_keys = set(train_pool['description_key']) & validation_keys
train_pool = train_pool.loc[~train_pool['description_key'].isin(cross_split_keys)].copy()
assert set(train_pool['description_key']).isdisjoint(validation_keys)
assert train_pool.index.intersection(validation_all.index).empty
assert all(train_pool['color'].eq(color).sum() >= 500 for color in LABELS)

ordered_train = {
    color: train_pool.loc[train_pool['color'].eq(color)].sample(n=500, random_state=SEED)
    for color in LABELS
}
train_sample = pd.concat([ordered_train[color] for color in LABELS]).sample(frac=1, random_state=SEED)
assert len(train_sample) == N_TRAIN
assert set(train_sample['description_key']).isdisjoint(validation_keys)
print('Cross-split normalized descriptions excluded from training:', len(cross_split_keys))
display(pd.DataFrame({'training': train_sample['color'].value_counts(), 'validation': validation_all['color'].value_counts()}).reindex(LABELS))
print('Validation uses every eligible California red/white review:', len(validation_all))

Cross-split normalized descriptions excluded from training: 0


,training,validation
color,,
red,500,6791
white,500,2614


Validation uses every eligible California red/white review: 9405


## 3. Compare a dummy reference and three TF-IDF classifiers

The same text representation, train rows, and validation rows are used for Naive Bayes, logistic regression, and SVM. The dummy model ignores text.

In [3]:
def text_features():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=10_000, sublinear_tf=True)

models = {
    'Majority baseline': DummyClassifier(strategy='most_frequent'),
    'Naive Bayes': make_pipeline(text_features(), MultinomialNB(alpha=1.0)),
    'Logistic regression': make_pipeline(text_features(), LogisticRegression(C=1.0, max_iter=2000, random_state=SEED)),
    'Linear SVM': make_pipeline(text_features(), LinearSVC(C=1.0, max_iter=5000, random_state=SEED)),
}
X_train, y_train = train_sample['description'], train_sample['color']
X_val, y_val = validation_all['description'], validation_all['color']
score_rows = []
validation_predictions = {}

def record_scores(name, train_true, train_pred, val_true, val_pred):
    for split, actual, predicted in [('training', train_true, train_pred), ('validation', val_true, val_pred)]:
        score_rows.append({'model': name, 'split': split, 'n': len(actual), 'accuracy': accuracy_score(actual, predicted), 'macro_f1': f1_score(actual, predicted, labels=LABELS, average='macro', zero_division=0)})
    validation_predictions[name] = np.asarray(val_pred)

for name, model in models.items():
    model.fit(X_train, y_train)
    record_scores(name, y_train, model.predict(X_train), y_val, model.predict(X_val))
    print('Finished', name)

Finished Majority baseline
Finished Naive Bayes
Finished Logistic regression
Finished Linear SVM


In [4]:
def show_comparison():
    scores = pd.DataFrame(score_rows).pivot(index='model', columns='split', values=['accuracy', 'macro_f1'])
    scores.columns = [f'{metric}_{split}' for metric, split in scores.columns]
    scores['accuracy_gap'] = scores['accuracy_training'] - scores['accuracy_validation']
    scores['macro_f1_gap'] = scores['macro_f1_training'] - scores['macro_f1_validation']
    display(scores.sort_values('macro_f1_validation', ascending=False).round(3))
    for name, predicted in validation_predictions.items():
        print(f'\n{name}: validation report')
        print(classification_report(y_val, predicted, labels=LABELS, zero_division=0, digits=3))
        display(pd.DataFrame(confusion_matrix(y_val, predicted, labels=LABELS), index=['actual red', 'actual white'], columns=['predicted red', 'predicted white']))
    return scores

comparison = show_comparison()

,accuracy_training,accuracy_validation,macro_f1_training,macro_f1_validation,accuracy_gap,macro_f1_gap
model,,,,,,
Linear SVM,1.000,0.982,1.000,0.978,0.018,0.022
Logistic regression,1.000,0.981,1.000,0.976,0.019,0.024
Naive Bayes,0.996,0.979,0.996,0.974,0.017,0.022
Majority baseline,0.500,0.722,0.333,0.419,-0.222,-0.086



Majority baseline: validation report
              precision    recall  f1-score   support

         red      0.722     1.000     0.839      6791
       white      0.000     0.000     0.000      2614

    accuracy                          0.722      9405
   macro avg      0.361     0.500     0.419      9405
weighted avg      0.521     0.722     0.606      9405



,predicted red,predicted white
actual red,6791,0
actual white,2614,0



Naive Bayes: validation report
              precision    recall  f1-score   support

         red      0.990     0.981     0.985      6791
       white      0.951     0.974     0.962      2614

    accuracy                          0.979      9405
   macro avg      0.970     0.977     0.974      9405
weighted avg      0.979     0.979     0.979      9405



,predicted red,predicted white
actual red,6660,131
actual white,68,2546



Logistic regression: validation report
              precision    recall  f1-score   support

         red      0.992     0.981     0.987      6791
       white      0.953     0.979     0.966      2614

    accuracy                          0.981      9405
   macro avg      0.972     0.980     0.976      9405
weighted avg      0.981     0.981     0.981      9405



,predicted red,predicted white
actual red,6665,126
actual white,55,2559



Linear SVM: validation report
              precision    recall  f1-score   support

         red      0.992     0.983     0.987      6791
       white      0.957     0.980     0.968      2614

    accuracy                          0.982      9405
   macro avg      0.974     0.981     0.978      9405
weighted avg      0.982     0.982     0.982      9405



,predicted red,predicted white
actual red,6675,116
actual white,53,2561


### Observed local TF-IDF results

These results were reproduced from the code above on **1,000 balanced training reviews** and **all 9,405 eligible California red/white validation reviews** (6,791 red; 2,614 white). DistilBERT has not been run here.

| Model | Training accuracy | Validation accuracy | Validation macro F1 |
|---|---:|---:|---:|
| Naive Bayes | 99.6% | 97.9% | 0.974 |
| Logistic regression | 100.0% | 98.1% | 0.976 |
| Linear SVM | 100.0% | 98.2% | 0.978 |

The majority baseline gets **72.2%** validation accuracy and **0.419** macro F1 because red reviews are the majority in this population. The near-perfect training scores and roughly 1.7–1.9 percentage-point validation gaps suggest mild overfitting; inspect which varietals are hardest before changing regularization or adding data. These validation scores use a broader population than Model 4, so compare them as a scope test, not as a direct learning-curve step.

### Bias–variance decisions for the four models

- **Naive Bayes:** If both training and validation macro F1 are low, test feature settings or smoothing (`alpha`). If training exceeds validation markedly, simplify features or add representative training data.
- **Logistic regression:** If both scores are low, inspect words and try a larger `C` only for a clear underfitting pattern. If the gap is large, use a smaller `C` or more data. Coefficients can help explain review language.
- **Linear SVM:** A large training–validation gap suggests stronger regularization (smaller `C`) or more representative data. Low scores on both splits suggest feature or label limitations. Inspect mistakes by varietal rather than relying on accuracy alone.
- **DistilBERT:** If it overfits, try more data or fewer epochs. If both scores are low, check labels and training setup. Compare its validation macro F1 with the simpler models before spending more compute.

The full California validation set is naturally imbalanced, so report **macro F1 and per-class precision/recall** alongside accuracy. Training–validation gaps are diagnostic clues, not formal bias/variance estimates. The larger validation set makes these estimates more stable than Model 4's 2,922-review set.

## 4. Fine-tune DistilBERT on the same rows

A Colab GPU is recommended because backpropagation through DistilBERT is much more expensive than sparse TF-IDF fitting. Colab is for compute and setup convenience; it does not change the evaluation design. The setup cell below lists installed versions. If a package is missing or incompatible, uncomment its `%pip` line, run it once, **restart the kernel/runtime**, then rerun the notebook from the top. Do not upgrade packages and import them in the same live kernel.

In [5]:
# Check the packages used by the DistilBERT section.
# If an import fails, uncomment and run the next line ONCE, then restart the kernel.
# %pip -q install -U "transformers==4.57.6" "accelerate>=1,<2" "datasets>=4,<5"
from importlib import metadata
for package in ('torch', 'transformers', 'accelerate', 'datasets', 'pyarrow'):
    try:
        print(f'{package}: {metadata.version(package)}')
    except metadata.PackageNotFoundError:
        print(f'{package}: missing; install it and restart the kernel')

torch: 2.12.0
transformers: 4.57.6
accelerate: 1.15.0
datasets: 4.8.5
pyarrow: 23.0.1


In [6]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments, set_seed


set_seed(SEED)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else ('Apple MPS' if torch.backends.mps.is_available() else 'CPU'))
MODEL_NAME = 'distilbert/distilbert-base-uncased'
label2id = {'red': 0, 'white': 1}
id2label = {v: k for k, v in label2id.items()}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, frame):
        self.encodings = tokenizer(frame['description'].tolist(), truncation=True, max_length=128)
        self.labels = frame['color'].map(label2id).astype(int).tolist()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return {**{key: torch.tensor(values[index]) for key, values in self.encodings.items()}, 'labels': torch.tensor(self.labels[index])}

bert_train = ReviewDataset(train_sample)
bert_val = ReviewDataset(validation_all)

/opt/anaconda3/envs/DSgeneral/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: Apple MPS


In [7]:
def bert_metrics(result):
    predicted = result.predictions.argmax(axis=-1)
    return {'accuracy': accuracy_score(result.label_ids, predicted), 'macro_f1': f1_score(result.label_ids, predicted, labels=[0, 1], average='macro', zero_division=0)}

bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id)
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported()
bert_args = TrainingArguments(
    output_dir=f'modeling_{MODEL_NUMBER}_bert_checkpoints',
    learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=3, weight_decay=0.01,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='macro_f1', greater_is_better=True, save_total_limit=1,
    fp16=torch.cuda.is_available() and not use_bf16, bf16=use_bf16,
    report_to='none', seed=SEED,
)
bert_trainer = Trainer(
    model=bert_model, args=bert_args, train_dataset=bert_train, eval_dataset=bert_val,
    processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=bert_metrics,
)
bert_trainer.train()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/anaconda3/envs/DSgeneral/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.153748,0.966188,0.956970
2,No log,0.176547,0.938224,0.927199
3,No log,0.078400,0.975864,0.970269


/opt/anaconda3/envs/DSgeneral/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/envs/DSgeneral/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=189, training_loss=0.23382836801034432, metrics={'train_runtime': 712.4118, 'train_samples_per_second': 4.211, 'train_steps_per_second': 0.265, 'total_flos': 72341688287904.0, 'train_loss': 0.23382836801034432, 'epoch': 3.0})

In [8]:
train_output = bert_trainer.predict(bert_train)
val_output = bert_trainer.predict(bert_val)
bert_train_pred = np.array([id2label[int(i)] for i in train_output.predictions.argmax(axis=-1)])
bert_val_pred = np.array([id2label[int(i)] for i in val_output.predictions.argmax(axis=-1)])
record_scores('DistilBERT', y_train, bert_train_pred, y_val, bert_val_pred)
comparison = show_comparison()
print('Selected checkpoint:', bert_trainer.state.best_model_checkpoint)

/opt/anaconda3/envs/DSgeneral/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/opt/anaconda3/envs/DSgeneral/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


,accuracy_training,accuracy_validation,macro_f1_training,macro_f1_validation,accuracy_gap,macro_f1_gap
model,,,,,,
Linear SVM,1.000,0.982,1.000,0.978,0.018,0.022
Logistic regression,1.000,0.981,1.000,0.976,0.019,0.024
Naive Bayes,0.996,0.979,0.996,0.974,0.017,0.022
DistilBERT,0.992,0.976,0.992,0.970,0.016,0.022
Majority baseline,0.500,0.722,0.333,0.419,-0.222,-0.086



Majority baseline: validation report
              precision    recall  f1-score   support

         red      0.722     1.000     0.839      6791
       white      0.000     0.000     0.000      2614

    accuracy                          0.722      9405
   macro avg      0.361     0.500     0.419      9405
weighted avg      0.521     0.722     0.606      9405



,predicted red,predicted white
actual red,6791,0
actual white,2614,0



Naive Bayes: validation report
              precision    recall  f1-score   support

         red      0.990     0.981     0.985      6791
       white      0.951     0.974     0.962      2614

    accuracy                          0.979      9405
   macro avg      0.970     0.977     0.974      9405
weighted avg      0.979     0.979     0.979      9405



... (4 outputs left out)

,predicted red,predicted white
actual red,6675,116
actual white,53,2561



DistilBERT: validation report
              precision    recall  f1-score   support

         red      0.990     0.976     0.983      6791
       white      0.940     0.975     0.957      2614

    accuracy                          0.976      9405
   macro avg      0.965     0.976     0.970      9405
weighted avg      0.976     0.976     0.976      9405



,predicted red,predicted white
actual red,6629,162
actual white,65,2549


Selected checkpoint: modeling_5_bert_checkpoints/checkpoint-189


## Result and next decision

Looks like the previosu models all generalized well to the expanded scope. Still some minor overfit, but that is to be expected. next step is to expand to 10,000 training obs, and make final modelling decisions from there before going to fully scoped dataset. 